# Assignment 2 Notebook 02 
## Aim: Perform MQTT Subscription and support Dynamic Map Dashboard

In [11]:
# Install required packages from requirements.txt
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
requirements_path = PROJECT_ROOT / "requirements.txt"

if requirements_path.exists():
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_path)
    ])
    print("Packages installed from:", requirements_path)
else:
    raise FileNotFoundError(f"requirements.txt not found at: {requirements_path}")

Packages installed from: C:\Users\The_Won\Comp5339A2\requirements.txt


In [12]:
import json
import time

import folium
import pandas as pd
import paho.mqtt.client as mqtt
from IPython.display import display, clear_output, Markdown, HTML
import ipywidgets as widgets
import os

from src.mqtt_helpers import BROKER, PORT, TOPIC, make_mqtt_client, load_stream_csv, merge_market

stream_path = PROJECT_ROOT / "data" / "processed" / "facility_power_emissions_5min_may2026.csv"
market_path = PROJECT_ROOT / "data" / "processed" / "nem_market_price_demand_5min_may2026.csv"

print("Broker:", BROKER)
print("Topic:", TOPIC)
print("Project root:", PROJECT_ROOT)

Broker: broker.hivemq.com
Topic: comp5339/2026s1/team_xinyi/electricity/facilities
Project root: C:\Users\The_Won\Comp5339A2


In [13]:
# Metadata fallback is used only before live MQTT messages arrive, so the map can be tested offline.
base_stream_df = merge_market(load_stream_csv(stream_path), market_path)
base_latest_df = (
    base_stream_df.sort_values("timestamp")
    .drop_duplicates(subset=["facility_code"], keep="last")
    .reset_index(drop=True)
)

print("Offline fallback facilities:", len(base_latest_df))
display(base_latest_df.head())

Offline fallback facilities: 353


,timestamp,facility_code,facility_name,a1_facility_name,a1_match_score,state,network_region,facility_type,latitude,longitude,...,longitude_api,latitude_a1,longitude_a1,power_mw,emissions_t_co2e,registered_capacity_mw,installed_capacity_mw_a1,reporting_entity,price_aud_per_mwh,demand_mw
0,2026-05-07 23:55:00+10:00,OSBORNE,Osborne,Osborne facility,1.000000,SA,SA1,gas_ccgt,-34.785928,138.507289,...,138.507790,-34.785928,138.507289,0.00,0.0,180.00,3.091,ATCO AUSTRALIA PTY LTD,52.082,21627.0
1,2026-05-07 23:55:00+10:00,PALOONA,Paloona,NaN,0.700000,TAS,TAS1,hydro,-41.282674,146.249036,...,146.249036,NaN,NaN,29.79,0.0,28.00,NaN,NaN,52.082,21627.0
2,2026-05-07 23:55:00+10:00,PORTLCN,Port Lincoln,Port Lincoln Power Station,1.000000,SA,SA1,distillate,-34.700409,135.804504,...,135.804384,-34.700409,135.804504,0.00,0.0,73.50,3.091,International Power (Australia) Holdings Pty L...,52.082,21627.0
3,2026-05-07 23:55:00+10:00,PIBESS,Phillip Island,NaN,0.500000,VIC,VIC1,"battery, battery_charging, battery_discharging",-38.483293,145.224880,...,145.224880,NaN,NaN,0.00,0.0,21.00,NaN,NaN,52.082,21627.0
4,2026-05-07 23:55:00+10:00,PIONEER,Pioneer Sugar Mill,NaN,0.528302,QLD,QLD1,bioenergy_biomass,-19.557546,147.330825,...,147.330825,NaN,NaN,-0.40,0.0,67.78,NaN,NaN,52.082,21627.0


In [14]:
received_messages = []

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Subscriber connected.")
        client.subscribe(TOPIC, qos=1)
    else:
        print("Subscriber connection failed:", rc)

def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode("utf-8"))
        received_messages.append(payload)
    except Exception as exc:
        print("Failed to decode MQTT message:", exc)

subscriber = make_mqtt_client(f"comp5339_dashboard_subscriber_{int(time.time())}")
subscriber.on_connect = on_connect
subscriber.on_message = on_message
subscriber.connect(BROKER, PORT, keepalive=60)
subscriber.loop_start()

print("Subscriber started. Current received messages:", len(received_messages))

Subscriber started. Current received messages: 0
Subscriber connected.


In [15]:
# The notebook prototype starts from an empty live-message buffer.
# Set this to True only when you want to test the notebook map with cached CSV fallback.
USE_OFFLINE_FALLBACK = False

def current_dashboard_df() -> pd.DataFrame:
    if received_messages:
        df = pd.DataFrame(received_messages)
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        for col in ["power_mw", "emissions_t_co2e", "latitude", "longitude", "price_aud_per_mwh", "demand_mw"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.dropna(subset=["timestamp", "facility_code", "latitude", "longitude"])
        return (
            df.sort_values("timestamp")
            .drop_duplicates(subset=["facility_code"], keep="last")
            .reset_index(drop=True)
        )
    if USE_OFFLINE_FALLBACK:
        return base_latest_df.copy()

    return pd.DataFrame(columns=base_latest_df.columns)

import math

def build_facility_map(dataframe: pd.DataFrame, metric: str, states: tuple[str, ...], facility_types: tuple[str, ...]) -> folium.Map:
    df = dataframe.copy()

    if states:
        df = df[df["state"].isin(states)]

    if facility_types:
        df = df[df["facility_type"].isin(facility_types)]

    df = df.dropna(subset=["latitude", "longitude"])

    if df.empty:
        return folium.Map(location=[-25.5, 134.5], zoom_start=4, tiles="OpenStreetMap")

    m = folium.Map(
        location=[df["latitude"].mean(), df["longitude"].mean()],
        zoom_start=5,
        tiles="OpenStreetMap"
    )

    # Use different marker-size source values for each map type.
    if metric == "power_mw":
        radius_col = "power_mw"
        color = "#2563eb"
        metric_label = "Power"
        unit_label = "MW"
    elif metric == "emissions_t_co2e":
        radius_col = "emissions_t_co2e"
        color = "#dc2626"
        metric_label = "Emissions"
        unit_label = "tCO2e"
    else:
        radius_col = metric
        color = "#374151"
        metric_label = metric
        unit_label = ""

    # Avoid negative / missing values dominating marker size.
    radius_values = pd.to_numeric(df[radius_col], errors="coerce").fillna(0).clip(lower=0)

    # Square-root scaling makes differences easier to see without letting large facilities dominate.
    max_radius_value = max(float(radius_values.max()), 1.0)

    for _, row in df.iterrows():
        radius_value = max(float(row.get(radius_col, 0) or 0), 0)

        if metric == "power_mw":
            radius = 4 + 20 * math.sqrt(radius_value / max_radius_value)
            radius = min(max(radius, 4), 24)
        else:
            radius = 3 + 13 * math.sqrt(radius_value / max_radius_value)
            radius = min(max(radius, 3), 16)

        popup_html = f"""
        <div style='width: 300px'>
            <h4>{row.get('facility_name', 'Unknown')}</h4>
            <b>Code:</b> {row.get('facility_code', '')}<br>
            <b>Type:</b> {row.get('facility_type', '')}<br>
            <b>State/Region:</b> {row.get('state', '')} / {row.get('network_region', '')}<br>
            <b>Timestamp:</b> {row.get('timestamp', '')}<br>
            <b>Power:</b> {row.get('power_mw', 0):,.2f} MW<br>
            <b>Emissions:</b> {row.get('emissions_t_co2e', 0):,.2f} tCO2e<br>
            <b>NEM price:</b> {row.get('price_aud_per_mwh', float('nan')):,.2f} $/MWh<br>
            <b>NEM demand:</b> {row.get('demand_mw', float('nan')):,.2f} MW
        </div>
        """

        tooltip = (
            f"{row.get('facility_name')} | "
            f"{metric_label}: {row.get(metric, 0):,.2f} {unit_label}"
        )

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.72,
            popup=folium.Popup(popup_html, max_width=340),
            tooltip=tooltip,
        ).add_to(m)

    return m

## Prototype Notebook Dashboard

This Folium + ipywidgets dashboard is used to demonstrate live MQTT subscription from an empty state. By default it does not use the cached CSV fallback, so the map starts with no facility markers and gradually updates as MQTT messages are received from the publisher in Notebook 01. The final Streamlit dashboard later in this notebook uses the cached CSV fallback to provide a stable full-map user interface.


In [16]:
metric_selector = widgets.ToggleButtons(
    options=[("Power MW", "power_mw"), ("Emissions tCO2e", "emissions_t_co2e")],
    value="power_mw",
    description="Metric"
)

initial_df = base_latest_df.copy()
state_selector = widgets.SelectMultiple(
    options=sorted(initial_df["state"].dropna().astype(str).unique().tolist()),
    value=tuple(sorted(initial_df["state"].dropna().astype(str).unique().tolist())),
    description="States"
)

type_selector = widgets.SelectMultiple(
    options=sorted(initial_df["facility_type"].dropna().astype(str).unique().tolist()),
    value=tuple(sorted(initial_df["facility_type"].dropna().astype(str).unique().tolist())),
    description="Types"
)

refresh_button = widgets.Button(description="Refresh map", button_style="primary")
output = widgets.Output()

def redraw(_=None):
    df = current_dashboard_df()
    latest_timestamp = df["timestamp"].max() if not df.empty else "No live data yet"
    latest_price = df["price_aud_per_mwh"].dropna().iloc[-1] if "price_aud_per_mwh" in df and df["price_aud_per_mwh"].notna().any() else None
    latest_demand = df["demand_mw"].dropna().iloc[-1] if "demand_mw" in df and df["demand_mw"].notna().any() else None
    with output:
        clear_output(wait=True)
        display(Markdown(f"**Messages received:** {len(received_messages)}  |  **Facilities currently shown:** {df['facility_code'].nunique() if not df.empty else 0}  |  **Latest event time:** {latest_timestamp}  |  **Price:** {latest_price if latest_price is not None else 'N/A'}  |  **Demand:** {latest_demand if latest_demand is not None else 'N/A'}"))
        display(build_facility_map(df, metric_selector.value, tuple(state_selector.value), tuple(type_selector.value)))

refresh_button.on_click(redraw)
metric_selector.observe(redraw, names="value")
state_selector.observe(redraw, names="value")
type_selector.observe(redraw, names="value")

display(widgets.VBox([widgets.HBox([metric_selector, refresh_button]), widgets.HBox([state_selector, type_selector]), output]))
redraw()

### This chunk demonstrate the full-facility power dashboard map from all cached data.

In [17]:
# Generate and save a full-facility dashboard map from all cached data.
# This uses the latest record for every facility in the processed CSV,

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

full_dashboard_df = base_latest_df.copy()

full_map = build_facility_map(
    dataframe=full_dashboard_df,
    metric="power_mw",  # Change to "emissions_t_co2e" if needed
    states=tuple(sorted(full_dashboard_df["state"].dropna().astype(str).unique())),
    facility_types=tuple(sorted(full_dashboard_df["facility_type"].dropna().astype(str).unique())),
)

full_map_path = OUTPUT_DIR / "mqtt_facility_dashboard_full_map.html"
full_map.save(full_map_path)

print("Full facility map saved to:", full_map_path)
print("Facilities shown:", full_dashboard_df["facility_code"].nunique())
print("Latest timestamp:", full_dashboard_df["timestamp"].max())

display(full_map)

Full facility map saved to: C:\Users\The_Won\Comp5339A2\output\mqtt_facility_dashboard_full_map.html
Facilities shown: 353
Latest timestamp: 2026-05-07 23:55:00+10:00


### This chunk demonstrate the full-facility emission dashboard map from all cached data.

In [18]:
full_emissions_map = build_facility_map(
    dataframe=full_dashboard_df,
    metric="emissions_t_co2e",
    states=tuple(sorted(full_dashboard_df["state"].dropna().astype(str).unique())),
    facility_types=tuple(sorted(full_dashboard_df["facility_type"].dropna().astype(str).unique())),
)

full_emissions_map_path = OUTPUT_DIR / "mqtt_facility_dashboard_full_emissions_map.html"
full_emissions_map.save(full_emissions_map_path)

print("Full emissions map saved to:", full_emissions_map_path)

display(full_emissions_map)

Full emissions map saved to: C:\Users\The_Won\Comp5339A2\output\mqtt_facility_dashboard_full_emissions_map.html


## Final Live Streamlit Dashboard

The Streamlit app subscribes to the same MQTT topic and uses the cached CSV as a fallback so the map is populated even before new live MQTT messages arrive. It supports power/emissions switching, state and facility-type filters, automatic refresh, current NEM price/demand display, clearing or pausing the live message buffer, and saving the current map as HTML.


In [19]:
# Open the live Streamlit dashboard instead of using the notebook auto-refresh loop.
# The full-data Folium HTML exports above are still kept for report screenshots/evidence.

import html
import socket
import subprocess

STREAMLIT_PORT = 8501
STREAMLIT_HOST = "localhost"
STREAMLIT_URL = f"http://{STREAMLIT_HOST}:{STREAMLIT_PORT}"
STREAMLIT_APP = PROJECT_ROOT / "streamlit_dashboard.py"
STREAMLIT_LOG = PROJECT_ROOT / "output" / "streamlit_dashboard_launch.log"


def is_port_open(host=STREAMLIT_HOST, port=STREAMLIT_PORT, timeout=1.0):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def read_log_tail(log_path: Path, max_chars: int = 4000) -> str:
    if not log_path.exists():
        return "No Streamlit log file was created."
    text = log_path.read_text(errors="replace")
    return text[-max_chars:] if text else "Streamlit log is empty."


if not STREAMLIT_APP.exists():
    raise FileNotFoundError(f"Streamlit app not found: {STREAMLIT_APP}")

STREAMLIT_LOG.parent.mkdir(parents=True, exist_ok=True)
launch_status = "already_running"
launch_error = ""

if not is_port_open():
    creationflags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
    with open(STREAMLIT_LOG, "w", encoding="utf-8") as log_file:
        streamlit_process = subprocess.Popen(
            [
                sys.executable,
                "-m",
                "streamlit",
                "run",
                str(STREAMLIT_APP),
                "--server.port",
                str(STREAMLIT_PORT),
                "--server.address",
                STREAMLIT_HOST,
                "--server.headless",
                "true",
                "--browser.gatherUsageStats",
                "false",
            ],
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            creationflags=creationflags,
        )

    launch_status = "starting"
    for _ in range(20):
        if is_port_open():
            launch_status = "started"
            break
        exit_code = streamlit_process.poll()
        if exit_code is not None:
            launch_status = "failed"
            launch_error = read_log_tail(STREAMLIT_LOG)
            break
        time.sleep(1)

    if launch_status == "starting":
        launch_status = "failed"
        launch_error = (
            "Streamlit did not start listening on the expected port within 20 seconds.\n\n"
            + read_log_tail(STREAMLIT_LOG)
        )
else:
    print("Streamlit dashboard is already running.")

if launch_status in {"started", "already_running"}:
    if launch_status == "started":
        print("Streamlit dashboard started successfully.")

    display(HTML(f"""
<div style="font-family: Arial; border: 1px solid #d0d7de; padding: 14px; border-radius: 6px; max-width: 760px;">
  <h3 style="margin: 0 0 8px 0;">Live Streamlit Dashboard</h3>
  <p style="margin: 0 0 12px 0;">
    Use the Streamlit dashboard for live MQTT subscription, automatic refresh, metric switching,
    state/type filters, and current price/demand display.
  </p>
  <a href="{STREAMLIT_URL}" target="_blank"
     style="display: inline-block; background: #2563eb; color: white; padding: 8px 12px; border-radius: 4px; text-decoration: none;">
     Open Streamlit Dashboard
  </a>
  <p style="margin: 12px 0 0 0; color: #57606a; font-size: 13px;">
    URL: {STREAMLIT_URL}<br>
    Log: {STREAMLIT_LOG}
  </p>
</div>
"""))
else:
    escaped_error = html.escape(launch_error)
    print("Streamlit dashboard failed to start. Review the log below.")
    display(HTML(f"""
<div style="font-family: Arial; border: 1px solid #f5c2c7; background: #fff5f5; padding: 14px; border-radius: 6px; max-width: 900px;">
  <h3 style="margin: 0 0 8px 0; color: #b42318;">Streamlit failed to start</h3>
  <p style="margin: 0 0 12px 0;">
    The dashboard link is not shown because no service is listening on <code>{STREAMLIT_URL}</code> yet.
  </p>
  <p style="margin: 0 0 12px 0; color: #57606a; font-size: 13px;">
    Log file: {STREAMLIT_LOG}
  </p>
  <pre style="white-space: pre-wrap; margin: 0; font-size: 12px; background: white; border: 1px solid #f1b0b7; padding: 10px; border-radius: 4px;">{escaped_error}</pre>
</div>
"""))


Streamlit dashboard is already running.


## Optional Cleanup

Run this cell only to stop the notebook-based MQTT subscriber for the prototype dashboard. It does not affect the Streamlit dashboard, which uses its own subscriber. Remove the comment symbols before running this cell.

In [20]:
# Optional cleanup for the notebook MQTT subscriber.
# This does not stop the Streamlit dashboard.

'''
try:
    subscriber.loop_stop()
    subscriber.disconnect()
    print("Notebook MQTT subscriber stopped.")
except NameError:
    print("Notebook MQTT subscriber was not started.")
'''

Notebook MQTT subscriber stopped.
